In [164]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dataclasses import dataclass, field
from scipy.optimize import minimize
from filterpy.kalman import MerweScaledSigmaPoints, UnscentedKalmanFilter
from tqdm import tqdm

In [165]:
discounts = pd.read_excel('discounts.xlsx', index_col='date')
discounts = discounts.dropna()

panel_data = pd.read_excel('macrodata.xlsx', index_col='meeting_date')
panel_data = panel_data.dropna()
is_meeting_date = discounts.index.isin(panel_data.index)

In [166]:
def meeting_offsets_from_dates(meeting_dates, start_date, T_max, day_count='BUS/252'):
    md = pd.to_datetime(pd.Index(meeting_dates)).sort_values().unique()
    t0 = pd.to_datetime(start_date)

    if day_count.upper() == 'BUS/252':
        def yf(d): 
            return np.busday_count(t0.date(), pd.Timestamp(d).date()) / 252.0
    else: 
        def yf(d):
            return (pd.Timestamp(d) - t0).days / 365.0

    offs = np.array([yf(d) for d in md if pd.Timestamp(d) >= t0], dtype=float)
    offs = offs[(offs > 0) & (offs <= T_max + 1e-12)]
    return np.unique(np.round(offs, 10))

In [167]:
@dataclass
class JumpModelDiscount:    
    jump_size: int                  
    possible_jumps: np.ndarray                   
    meeting_dates: np.ndarray           
    T: float

    sigma0: float = 0.01                
    beta: float = 0.01                  
    mu_x: float = 0.0                   

    dt: float = 1/504                   
    x_min: float = -3.0
    x_max: float =  3.0
    r_min: float = 0.0
    r_max: float = 0.1
    Nx: int = 181

    softmax_temp: float = 1.0

    M: np.ndarray = field(default=None, init=False)     
    x_grid: np.ndarray = field(default=None, init=False)
    t_grid: np.ndarray = field(default=None, init=False)
    r_grid: np.ndarray = field(default=None, init=False)

    def __jump_probs(self, x_vec: np.ndarray) -> np.ndarray:
        """
        returns P shape (Nx, nJ): softmax over J_bps with score = (J / jump_size)*x*temp
        """
        scores = (self.possible_jumps[None, :] / self.jump_size) * x_vec[:, None] * self.softmax_temp
        scores = scores - scores.max(axis=1, keepdims=True)  # numeric stability
        e = np.exp(scores)
        return e / e.sum(axis=1, keepdims=True)

    # def __step_diffusion_discount(self, U: np.ndarray, r_k: float, dt: float) -> np.ndarray:
    #     x = self.x_grid
    #     dx = x[1] - x[0]
    #     sigma = self.sigma0 + self.beta * np.abs(x)
    #     sig2 = sigma**2

    #     U_new = np.empty_like(U)

    #     # interior stencils
    #     i = np.arange(1, self.Nx-1)
    #     ux  = (U[i+1] - U[i-1]) / (2.0*dx)
    #     uxx = (U[i+1] - 2.0*U[i] + U[i-1]) / dx**2
    #     Lx  = self.mu_x * ux + 0.5 * sig2[i] * uxx
    #     U_new[i] = U[i] + dt * (Lx - r_k * U[i])

    #     # Neumann at boundaries: mirror
    #     j0, jN = 0, self.Nx-1
    #     # left: u_x ≈ 0 ⇒ use U[1] as ghost
    #     ux_left  = 0.0
    #     uxx_left = (U[1] - 2.0*U[0] + U[1]) / dx**2
    #     Lx_left  = self.mu_x * ux_left + 0.5 * sig2[j0] * uxx_left
    #     U_new[j0] = U[j0] + dt * (Lx_left - r_k * U[j0])

    #     # right: u_x ≈ 0 ⇒ mirror U[N-2]
    #     ux_right  = 0.0
    #     uxx_right = (U[jN-1] - 2.0*U[jN] + U[jN-1]) / dx**2
    #     Lx_right  = self.mu_x * ux_right + 0.5 * sig2[jN] * uxx_right
    #     U_new[jN] = U[jN] + dt * (Lx_right - r_k * U[jN])
        
    #     return U_new

    def __apply_jump(self, M_after: np.ndarray) -> np.ndarray:
        Nx, Nr = M_after.shape
        rg = self.r_grid
        Mplus = M_after

        r_target = rg[:, None] + self.possible_jumps[None, :]     

        idx_right = np.searchsorted(rg, r_target, side='left')     
        idx_right = np.clip(idx_right, 1, Nr - 1)                  
        idx_left  = idx_right - 1

        r_left  = rg[idx_left]                                     
        r_right = rg[idx_right]                                    

        w = (r_target - r_left) / (r_right - r_left)       
        P = self.__jump_probs(self.x_grid)

        Mminus = np.zeros_like(Mplus)
        nJ = self.possible_jumps.size
        for j in range(nJ):
            M_lr = (1.0 - w[:, j])[None, :] * Mplus[:, idx_left[:, j]] + w[:, j][None, :] * Mplus[:, idx_right[:, j]]   
            Mminus += P[:, j][:, None] * M_lr

        return Mminus


        # --- tiny Thomas solver for a tridiagonal system ---
    
    @staticmethod
    def __solve_tridiag(l, d, u, b):
        """
        Solve tridiagonal system with lower diag l (N-1,), main d (N,),
        upper diag u (N-1,), right-hand side b (N,).
        Returns x (N,). Operates in-place on copies.
        """
        N = d.size
        lc = l.copy()
        dc = d.copy()
        uc = u.copy()
        bc = b.copy()

        # forward sweep
        for i in range(1, N):
            w = lc[i-1] / dc[i-1]
            dc[i]   = dc[i]   - w * uc[i-1]
            bc[i]   = bc[i]   - w * bc[i-1]

        # back substitution
        x = np.empty_like(bc)
        x[-1] = bc[-1] / dc[-1]
        for i in range(N-2, -1, -1):
            x[i] = (bc[i] - uc[i] * x[i+1]) / dc[i]
        return x

    def __step_diffusion_discount(self, U_old: np.ndarray, r_k: float, dt: float) -> np.ndarray:
        """
        One Crank–Nicolson step for:
            U_t = mu_x * U_x + 0.5*sigma(x)^2 * U_xx - r_k * U
        with Neumann BC: U_x(±)=0 (mirror ghost points).
        """
        x  = self.x_grid
        dx = x[1] - x[0]

        sigma = self.sigma0 + self.beta * np.abs(x)
        sig2  = sigma * sigma

        N = self.Nx
        C = self.mu_x / (2.0 * dx)          # convection multiplier
        D = 0.5 * sig2 / (dx * dx)          # diffusion multiplier (vectorized)

        # Build L in tri-diagonal form: L U = lowerL*U_{i-1} + diagL*U_i + upperL*U_{i+1}
        lowerL = np.zeros(N-1, dtype=float)
        upperL = np.zeros(N-1, dtype=float)
        diagL  = np.zeros(N,   dtype=float)

        # interior nodes i=1..N-2 (central)
        i = np.arange(1, N-1)
        lowerL[i-1] = (D[i] - C)                  # U_{i-1}
        upperL[i-1] = (D[i] + C)                  # U_{i+1}
        diagL[i]    = (-2.0 * D[i] - r_k)         # U_i

        # Neumann left boundary: U_x=0 => U_{-1}=U_{1}
        # 0.5*sigma^2 * U_xx ≈ (sig2/dx^2)*(U1 - U0) = D_b*(U1 - U0)*2? careful:
        # Using D = 0.5*sig2/dx^2, the boundary diffusion becomes 2*D(0)*(U1 - U0)
        Db0      = 2.0 * D[0]
        diagL[0] = (-Db0 - r_k)
        upperL[0]= ( Db0 )

        # Neumann right boundary: mirror U_{N}=U_{N-2}
        DbN        = 2.0 * D[-1]
        diagL[-1]  = (-DbN - r_k)
        lowerL[-1] = ( DbN )

        # Crank–Nicolson matrices: (I - 0.5 dt L) U^{n+1} = (I + 0.5 dt L) U^{n}
        half = 0.5 * dt

        lowerA = -half * lowerL
        diagA  =  1.0 - half * diagL
        upperA = -half * upperL

        lowerB =  half * lowerL
        diagB  =  1.0 + half * diagL
        upperB =  half * upperL

        # RHS b = B * U_old (tridiagonal matvec)
        b = self.__tridiag_matvec(lowerB, diagB, upperB, U_old)

        # Solve A * U_new = b
        U_new = self.__solve_tridiag(lowerA, diagA, upperA, b)

        # Keep in (0,1] to avoid tiny drift (discount factors are ≤1 and >0)
        return U_new



    # def comp_grid(self):
    #     """
    #     Builds x/t grids, marches backward from T to 0.
    #     Stores P(0, x, r_k) in self.M with shape (Nx, Nr).
    #     """        
    #     self.x_grid = np.linspace(self.x_min, self.x_max, self.Nx)
    #     self.r_grid = np.arange(self.r_min, self.r_max+self.jump_size, self.jump_size)
    #     Nr = self.r_grid.size

    #     Nt = int(np.ceil(self.T / self.dt))
    #     self.t_grid = np.linspace(self.T, 0.0, Nt+1)
    #     dt = self.t_grid[0] - self.t_grid[1]

    #     M = np.ones((self.Nx, Nr), dtype=float)
    #     M_old = np.empty_like(M)

    #     meeting_idx = set()
    #     for tmeet in self.meeting_dates:
    #         k = int(round((self.T - tmeet) / dt))
    #         k = np.clip(k, 0, Nt)
    #         meeting_idx.add(k)

    #     # CFL - as we use an explicit scheme for diffusion
    #     dx = self.x_grid[1] - self.x_grid[0]
    #     sig_max = (self.sigma0 + self.beta * np.abs(self.x_grid)).max()
    #     cfl = (sig_max**2) * dt / dx**2
    #     if cfl > 0.5:
    #         raise ValueError(f"[warn] explicit diffusion CFL ~ {cfl:.3f} > 0.5 — consider smaller dt or larger Nx")

    #     for n in range(Nt):
    #         for k, r_k in enumerate(self.r_grid):
    #             M_old[:, k] = self.__step_diffusion_discount(M[:, k], r_k, dt)

    #         step_ix = n + 1
    #         if step_ix in meeting_idx:
    #             M_post_reset = self.__reset_x_to_zero(M_old)
    #             M_old[:, :] = self.__apply_jump(M_post_reset)
    #         M, M_old = M_old, M

    #     self.M = M

    @staticmethod
    def __tridiag_matvec(l, d, u, x):
        """Compute y = T x for tridiagonal T with lower l (N-1,), diag d (N,), upper u (N-1,)."""
        N = d.size
        y = d * x
        y[1:] += l * x[:-1]
        y[:-1] += u * x[1:]
        return y


    def comp_grid(self):
        self.x_grid = np.linspace(self.x_min, self.x_max, self.Nx)
        self.r_grid = np.arange(self.r_min, self.r_max + self.jump_size, self.jump_size)
        Nr = self.r_grid.size

        Nt = int(np.ceil(self.T / self.dt))
        self.t_grid = np.linspace(self.T, 0.0, Nt + 1)
        dt = float(self.t_grid[0] - self.t_grid[1])

        M = np.ones((self.Nx, Nr), dtype=float)
        M_old = np.empty_like(M)

        meeting_idx = set()
        for tmeet in self.meeting_dates:
            k = int(round((self.T - float(tmeet)) / dt))
            k = np.clip(k, 0, Nt)
            meeting_idx.add(k)

        for n in range(Nt):
            # CN step for each r_k
            for k, r_k in enumerate(self.r_grid):
                M_old[:, k] = self.__step_diffusion_discount(M[:, k], float(r_k), dt)

            # expected jump at t_{n+1}: reset x→0 then apply jump operator
            step_ix = n + 1
            if step_ix in meeting_idx:
                M_post_reset = self.__reset_x_to_zero(M_old)
                M_old[:, :]  = self.__apply_jump(M_post_reset)

            M, M_old = M_old, M

        self.M = M


    def __reset_x_to_zero(self, M_layer: np.ndarray) -> np.ndarray:
        ix0 = int(np.argmin(np.abs(self.x_grid)))
        return np.broadcast_to(M_layer[ix0, :], (self.Nx, M_layer.shape[1])).copy()
    
    def discount(self, x: float, r: float, method: str = "nearest") -> float:
        if self.M is None:
            raise RuntimeError("Run comp_grid() first.")

        xg = self.x_grid
        if x <= xg[0]:
            ix0, wx = 0, 0.0
        elif x >= xg[-1]:
            ix0, wx = self.Nx-2, 1.0
        else:
            dx = xg[1] - xg[0]
            ix0 = int((x - xg[0]) / dx)
            ix0 = np.clip(ix0, 0, self.Nx-2)
            wx = (x - xg[ix0]) / dx

        rg = self.r_grid
        if method == "nearest":
            ik = int(np.clip(np.searchsorted(rg, r), 1, rg.size-1))
            ik = ik if abs(rg[ik]-r) <= abs(r-rg[ik-1]) else ik-1
            val0 = (1-wx)*self.M[ix0, ik] + wx*self.M[ix0+1, ik]
            return float(val0)
        elif method == "linear_r":
            ik = int(np.clip(np.searchsorted(rg, r), 1, rg.size-1))
            k0, k1 = ik-1, ik
            w = (r - rg[k0]) / (rg[k1] - rg[k0] + 1e-16)
            v0 = (1-wx)*self.M[ix0, k0] + wx*self.M[ix0+1, k0]
            v1 = (1-wx)*self.M[ix0, k1] + wx*self.M[ix0+1, k1]
            return float((1-w)*v0 + w*v1)
        else:
            raise ValueError("method must be 'nearest' or 'linear_r'")

In [168]:
jump = 25/10_000
possible_jumps = np.arange(-4*jump, 5*jump, jump)  
T_max = 2
meeting_dates = np.arange(0, T_max+1/12, 1/12)
model = JumpModelDiscount(
    jump_size=jump,
    possible_jumps=possible_jumps,
    meeting_dates=meeting_dates,    
    sigma0=0.01, beta=0.1,
    T=T_max, dt=1/25, Nx=25
)
model.comp_grid() 

x_axis = model.x_grid
r_axis_dec = model.r_grid
r_axis = r_axis_dec*100
P = model.M

fig_hm = go.Figure(
    data=go.Heatmap(
        x=x_axis,
        y=r_axis,
        z=P.T,               
        colorbar=dict(title="P(0,x,r)"),
        colorscale="Viridis",
        reversescale=False
    )
)
fig_hm.update_layout(
    title="Discount Surface P(0, x, r) — heatmap",
    xaxis_title="x",
    yaxis_title="r"
)
fig_hm.show()

fig_surf = go.Figure(
    data=go.Surface(
        x=x_axis,
        y=r_axis,
        z=P.T,
        colorbar=dict(title="P(0,x,r)"),
        colorscale="Viridis"
    )
)

fig_surf.update_layout(
    title="Discount Surface P(0, x, r) — 3D",
    scene=dict(
        xaxis_title="x",
        yaxis_title="r",
        zaxis_title="P(0,x,r)"
    ),
    height=650
)
fig_surf.show()

In [169]:
def periodic_meetings_from_gap(gap, T, period=0.25):
    gap = max(0.0, float(gap))
    if T <= 0:
        return np.array([], dtype=float)
    ks = np.arange(0, int(np.floor((T - gap)/period)) + 1, dtype=int)
    offs = gap + ks * period
    return offs[(offs > 0) & (offs <= T + 1e-12)].astype(float)

@dataclass
class ParameterMap:
    tenors: np.ndarray = field(default_factory=lambda: np.arange(0.5, 2.5, 0.5))

    s_min: float = 0.01
    s_max: float = 0.10
    dS: float    = 0.01
    b_min: float = -0.10
    b_max: float =  0.10
    dB: float    = 0.05

    period: float = 0.10
    gaps: np.ndarray = field(default_factory=lambda: np.linspace(0.0, 0.10, 10, endpoint=False))

    dt: float = 1/252
    Nx: int   = 50
    jump_size: float = 25/10_000
    possible_jumps: np.ndarray = None  

    # computed caches
    model_map: dict = field(default_factory=dict, init=False)  # gap -> s -> b -> list[models per tenor]
    s_values: np.ndarray = field(default=None, init=False)
    b_values: np.ndarray = field(default=None, init=False)

    def compute_grids(self):
        if self.possible_jumps is None:
            J = self.jump_size
            self.possible_jumps = np.arange(-4*J, 5*J, J)

        self.tenors = np.asarray(self.tenors, dtype=float)
        self.gaps   = np.asarray(self.gaps, dtype=float)

        s_range = np.arange(self.s_min, self.s_max + 1e-12, self.dS)
        b_range = np.arange(self.b_min, self.b_max + 1e-12, self.dB)
        self.s_values = s_range
        self.b_values = b_range

        # total number of grid solves = gaps × sigma × beta × tenor
        total_tasks = len(self.gaps) * len(s_range) * len(b_range) * len(self.tenors)
        pbar = tqdm(total=total_tasks, desc="⏳ Building Parameter Map", ncols=100)

        full_map = {}
        for g in self.gaps:
            by_s = {}
            for s in s_range:
                by_b = {}
                for b in b_range:
                    models_for_t = []
                    for T in self.tenors:
                        md_T = periodic_meetings_from_gap(gap=g, T=T, period=self.period)
                        m = JumpModelDiscount(
                            jump_size=self.jump_size,
                            possible_jumps=self.possible_jumps,
                            meeting_dates=md_T,
                            sigma0=s, beta=b, T=T,
                            dt=self.dt, Nx=self.Nx
                        )
                        m.comp_grid()
                        models_for_t.append(m)
                        pbar.update(1)
                    by_b[b] = models_for_t
                by_s[s] = by_b
            full_map[g] = by_s

        pbar.close()
        self.model_map = full_map

    def __pick_t_index(self, T):
        t = self.tenors
        idx = int(np.clip(np.searchsorted(t, T), 1, t.size - 1))
        if not np.isclose(t[idx], T):
            left = idx - 1
            idx = left if abs(t[left] - T) <= abs(t[idx] - T) else idx
        return idx

    def __interp_sb(self, models_at_gap, T_idx, x, sigma, beta, r0):
        s_idx = int(np.clip(np.searchsorted(self.s_values, sigma), 1, self.s_values.size - 1))
        b_idx = int(np.clip(np.searchsorted(self.b_values, beta),  1, self.b_values.size  - 1))
        s0, s1 = self.s_values[s_idx-1], self.s_values[s_idx]
        b0, b1 = self.b_values[b_idx-1], self.b_values[b_idx]

        ws = 0.0 if s1 == s0 else (sigma - s0) / (s1 - s0)
        wb = 0.0 if b1 == b0 else (beta  - b0) / (b1 - b0)

        m_s0b0 = models_at_gap[s0][b0][T_idx]
        m_s0b1 = models_at_gap[s0][b1][T_idx]
        m_s1b0 = models_at_gap[s1][b0][T_idx]
        m_s1b1 = models_at_gap[s1][b1][T_idx]

        d00 = m_s0b0.discount(x=x, r=r0, method="linear_r")
        d01 = m_s0b1.discount(x=x, r=r0, method="linear_r")
        d10 = m_s1b0.discount(x=x, r=r0, method="linear_r")
        d11 = m_s1b1.discount(x=x, r=r0, method="linear_r")

        d0 = (1 - ws) * d00 + ws * d10
        d1 = (1 - ws) * d01 + ws * d11
        return (1 - wb) * d0 + wb * d1

    def discount(self, T, x, sigma, beta, r0=0.0, gap_star=0.0):
        gi = int(np.clip(np.searchsorted(self.gaps, gap_star), 1, self.gaps.size - 1))
        g0, g1 = self.gaps[gi-1], self.gaps[gi]
        wg = 0.0 if g1 == g0 else (gap_star - g0) / (g1 - g0)
        T_idx = self.__pick_t_index(T)
        d0 = self.__interp_sb(self.model_map[g0], T_idx, x, sigma, beta, r0)
        d1 = self.__interp_sb(self.model_map[g1], T_idx, x, sigma, beta, r0)
        return (1 - wg) * d0 + wg * d1

    def discounts(self, x, sigma, beta, r0=0.0, gap_star=0.0):
        return np.array([self.discount(T, x, sigma, beta, r0=r0, gap_star=gap_star) for T in self.tenors])

In [170]:
param_map = ParameterMap(
    tenors=np.arange(0.25, 2.25, 0.25),
    s_min=0.01, s_max=0.5, dS=0.05,
    b_min=-0.1, b_max=0.1, dB=0.05,
    period=0.25,
    gaps=np.linspace(0.0, 0.25, 10, endpoint=False),
    dt=1/25, Nx=25,
    jump_size=25/10_000,
    possible_jumps=np.arange(-4*(25/10_000), 5*(25/10_000), (25/10_000))
)
param_map.compute_grids()

⏳ Building Parameter Map: 100%|████████████████████████████████| 4000/4000 [02:30<00:00, 26.66it/s]


In [171]:
x_range = np.linspace(-3.0, 3.0, 10)
beta_range = [param_map.b_min, 0.0, param_map.b_max]
sigma_range = [param_map.s_min, (param_map.s_min + param_map.s_max)/2, param_map.s_max]

for beta_hat in beta_range:
    for sigma_hat in sigma_range:
        fig = go.Figure()
        for x_val in x_range:
            t_discounts = param_map.discounts(
                x=x_val,
                sigma=sigma_hat,
                beta=beta_hat,
                r0=0.02,
                gap_star=0.0
            )
            fig.add_trace(
                go.Scatter(
                    x=param_map.tenors,
                    y=t_discounts,
                    mode='lines',
                    name=f"x={x_val:.2f}"
                )
            )
        fig.update_layout(
            title=f"Discount Curves for beta={beta_hat:.2f}, sigma={sigma_hat:.2f}",
            xaxis_title="Tenor T (years)",
            yaxis_title="Discount P(0,x,r=0,T)",
            height=600
        )
        fig.show()

In [172]:
def business_yf(start_date, end_date, bus_per_year=252):
    return np.busday_count(np.datetime64(start_date, 'D'), np.datetime64(end_date, 'D')) / bus_per_year

def gap_to_next_meeting(date_i, meeting_calendar):
    d = pd.to_datetime(date_i)
    mc = pd.to_datetime(pd.Index(meeting_calendar)).sort_values().unique()
    nxt = mc[mc > d]
    if nxt.size == 0:
        # fallback to one period if calendar exhausted
        period = getattr(param_map, 'period', 0.10)
        return period
    return business_yf(d.date(), pd.Timestamp(nxt[0]).date())


def calibrate_model(data: pd.DataFrame, meeting_flags, param_map: ParameterMap, alpha=1, beta=2, kappa=0, use_yields: bool = False):
    X = data.to_numpy()
    data_discounts = X[:, 1:]
    data_tenors = data.columns[1:]
    short_rate = (1/X[:, 0]-1) * 360.0  # keep your baseline

    N, M = data_discounts.shape[0], param_map.tenors.size
    obs_discounts = np.empty((N, M))
    for i in range(N):
        obs_discounts[i] = np.interp(param_map.tenors, data_tenors, data_discounts[i])

    measurements = (-np.log(obs_discounts) / param_map.tenors) if use_yields else obs_discounts
    obs_index = pd.to_datetime(data.index)
    meeting_calendar = obs_index[meeting_flags]
    gaps_per_obs = np.array([gap_to_next_meeting(d, meeting_calendar) for d in obs_index])
    dt = 1/252.0
    points = MerweScaledSigmaPoints(n=1, alpha=alpha, beta=beta, kappa=kappa)
    def obj_func(params):
        mu, sigma, beta = params
        def fx(x, dt_):
            return x-(sigma + beta * np.abs(x))*mu*dt_ 

        kf = UnscentedKalmanFilter(dim_x=1, dim_z=M, dt=dt, fx=fx, hx=None, points=points)

        kf.x = np.array([0.0])
        kf.P = np.array([[0.01]])

        z_std = 0.01
        kf.R = (z_std ** 2) * np.eye(M)
        total_loglike = 0.0
        for i in range(N):
            gap_i = float(gaps_per_obs[i])
            r0_i  = float(short_rate[i])
            
            def hx_regular(x):
                x_scalar = float(np.atleast_1d(x).item())
                pred_discounts = param_map.discounts(x=x_scalar, sigma=sigma, beta=beta,r0=r0_i, gap_star=gap_i)
                return (-np.log(pred_discounts) / param_map.tenors) if use_yields else pred_discounts

            if meeting_flags[i]:
                kf.x = np.array([0.0])
                kf.P = np.array([[0.01]])

            kf.hx = hx_regular
            sigmas = kf.points_fn.sigma_points(kf.x, kf.P)  
            local_sigma = sigma + beta * np.abs(sigmas[:, 0])
            q_var = np.dot(kf.points_fn.Wm, local_sigma * local_sigma) * dt
            kf.Q = np.array([[q_var]])

            z_t = measurements[i]   
            kf.predict()
            kf.update(z_t)
            total_loglike += float(kf.log_likelihood)
        
        print("Params:", params, " - LogLike:", -total_loglike)
        return -total_loglike
    return obj_func


In [173]:
sub_sample = discounts[4000:]
sub_meetings = is_meeting_date[4000:]
obj = calibrate_model(sub_sample, sub_meetings, param_map, use_yields=False)
res = minimize(obj, x0=np.array([0.01, 0.01, 0.01]), bounds=[(-2, 2), (param_map.s_min, param_map.s_max),(param_map.b_min, param_map.b_max)], method='L-BFGS-B')
print("theta* =", res.x, "  NLL =", res.fun)

Params: [0.01 0.01 0.01]  - LogLike: -11724.19148528043
Params: [0.01000001 0.01       0.01      ]  - LogLike: -11724.191485284227
Params: [0.01       0.01000001 0.01      ]  - LogLike: -11724.191517292616
Params: [0.01       0.01       0.01000001]  - LogLike: -11724.191529959468
Params: [0.38980498 0.5        0.1       ]  - LogLike: 302806.0373677383
Params: [0.38980499 0.5        0.1       ]  - LogLike: 303003.17429490044
Params: [0.38980498 0.49999999 0.1       ]  - LogLike: 295538.8116293691
Params: [0.38980498 0.5        0.09999999]  - LogLike: 307229.4676377468
Params: [0.1371928  0.17409598 0.04014008]  - LogLike: -5810.454486006909
Params: [0.13719281 0.17409598 0.04014008]  - LogLike: -5810.4544858768995
Params: [0.1371928  0.17409599 0.04014008]  - LogLike: -5810.453580026775
Params: [0.1371928  0.17409598 0.04014009]  - LogLike: -5810.449880230333
Params: [0.04045889 0.0492961  0.01721765]  - LogLike: -11910.659365536203
Params: [0.0404589  0.0492961  0.01721765]  - LogLike:

In [176]:
sub_sample = discounts[4000:]
sub_meetings = is_meeting_date[4000:]
X = sub_sample.to_numpy()
data_discounts = X[:, 1:]
data_tenors = discounts.columns[1:].to_numpy()
short_rate = (1/X[:, 0]-1) * 360.0

N = data_discounts.shape[0]
M = param_map.tenors.size

obs_discounts = np.vstack([
    np.interp(param_map.tenors, data_tenors, data_discounts[i])
    for i in range(N)
])

use_yields = True
measurements = (-np.log(obs_discounts) / param_map.tenors) if use_yields else obs_discounts

obs_index = pd.to_datetime(sub_sample.index)
meeting_calendar = pd.to_datetime(panel_data.index)
gaps_per_obs = np.array([gap_to_next_meeting(d, meeting_calendar) for d in obs_index], dtype=float)

mu_hat, sigma_hat, beta_hat = [0.01, 0.02, 0.01]  

dt = 1/252
points = MerweScaledSigmaPoints(n=1, alpha=1, beta=2, kappa=0)

def fx_mu(x, dt_):
    return x- (sigma_hat + beta_hat * np.abs(x)) * mu_hat * dt_

kf = UnscentedKalmanFilter(dim_x=1, dim_z=M, dt=dt,fx=fx_mu, hx=None, points=points)
kf.x = np.array([0.0])
kf.P = np.array([[0.01]])
z_std = 0.002
kf.R = (z_std**2) * np.eye(M)

x_filt = np.zeros(N)
fits_matrix = np.zeros((N, M))

for i in range(N):
    gap_i = gaps_per_obs[i]
    r0_i  = float(short_rate[i])
    
    def hx_regular(xvec):
        x_scalar = float(np.atleast_1d(xvec).item())
        pred_discounts = param_map.discounts(
            x=x_scalar, sigma=sigma_hat, beta=beta_hat,
            r0=r0_i, gap_star=gap_i
        )
        return -np.log(pred_discounts) / param_map.tenors if use_yields else pred_discounts

    if sub_meetings[i]:        
        kf.x = np.array([0.0])      
        kf.P = np.array([[0.01]])

    kf.hx = hx_regular
    sigmas_pts = kf.points_fn.sigma_points(kf.x, kf.P)
    local_sigma = sigma_hat + beta_hat * np.abs(sigmas_pts[:, 0])
    kf.Q = float(np.dot(kf.points_fn.Wm, local_sigma * local_sigma) * dt)
    
    z_t = measurements[i]
    kf.predict()
    kf.update(z_t)
    x_filt[i] = float(kf.x[0])
    fits_matrix[i, :] = hx_regular(kf.x)

tenors_to_plot = [0.5, 1.0, 2.0]
tenor_idx = [int(np.clip(np.searchsorted(param_map.tenors, t), 1, M-1)) for t in tenors_to_plot]
tenor_idx = [
    (idx-1 if abs(param_map.tenors[idx-1]-t)<=abs(param_map.tenors[idx]-t) else idx)
    for idx,t in zip(tenor_idx, tenors_to_plot)
]
tenor_labels = [f"{param_map.tenors[j]:.2f}y" for j in tenor_idx]
obs_series = measurements[:, tenor_idx]
fit_series = fits_matrix[:, tenor_idx]

In [177]:
fig_x = go.Figure()
fig_x.add_trace(go.Scatter(
    x=obs_index, y=x_filt, mode='lines',
    name='x(t)', line=dict(width=2)
))
fig_x.add_trace(go.Scatter(
    x=[obs_index[0], obs_index[-1]], y=[0, 0],
    mode='lines', name='zero line',
    line=dict(color='black', dash='dash')
))
fig_x.update_layout(
    title="Filtered State Evolution (x)",
    xaxis_title="Date",
    yaxis_title="x(t)",
    template="plotly_white",
    height=400
)
fig_x.show()

fig_curve = go.Figure()
for j, lbl in enumerate(tenor_labels):
    fig_curve.add_trace(go.Scatter(
        x=obs_index, y=obs_series[:, j],
        mode='lines', name=f'Observed {lbl}',
        line=dict(width=2)
    ))
    fig_curve.add_trace(go.Scatter(
        x=obs_index, y=fit_series[:, j],
        mode='lines', name=f'Fitted {lbl}',
        line=dict(width=2)
    ))

fig_curve.add_trace(go.Scatter(
    x=obs_index, y=short_rate,
    mode='lines', name='Short Rate (bps)',
    line=dict(width=2, color='orange')
))

fig_curve.update_layout(
    title=f"Observed vs Fitted {'Yields' if use_yields else 'Discounts'}",
    xaxis_title="Date",
    yaxis_title="Yield" if use_yields else "Discount",
    template="plotly_white",
    height=500
)
fig_curve.show()